# Project Summary: GaleMed Insights – A RAG-Based Medical Chatbot Using Pinecone and LLMs

---
## Project Overview
GaleMed Insights is an advanced medical chatbot built using a Retrieval-Augmented Generation (RAG) framework, leveraging the wealth of medical knowledge contained in The Gale Encyclopedia of Medicine, which spans over 637 pages. By combining PDF processing, semantic embedding, and vector databases, GaleMed Insights offers accurate, detailed, and user-friendly responses to medical inquiries. This makes it a valuable and accessible resource for individuals seeking trustworthy health information.

With its foundation in The Gale Encyclopedia of Medicine, GaleMed Insights ensures users can access well-researched, vetted content on a wide range of medical topics, from conditions and treatments to best healthcare practices. The chatbot empowers users to easily navigate complex medical concepts and obtain clear, relevant answers.

---

## Why Use RAG-Based Methodology Instead of Just LLMs?
Using a Retrieval-Augmented Generation (RAG) methodology offers significant advantages over traditional large language models (LLMs) alone:

#### Higher Accuracy and Reliability:
GaleMed Insights retrieves factual information directly from The Gale Encyclopedia of Medicine, ensuring that responses are factually accurate and relevant. This drastically reduces the risk of generating incorrect information (known as "hallucinations"), which is a common issue when using LLMs on their own.

#### Targeted Knowledge Base:
Instead of relying on broad datasets, GaleMed Insights focuses on a specific medical knowledge base, ensuring that users receive expert, specialized responses. This contrasts with general LLMs, which may provide superficial or irrelevant answers to health-related queries.

#### Contextual Relevance:
Through chunking and semantic embedding, GaleMed Insights retrieves the most relevant information while maintaining the necessary context, which leads to clearer and more coherent responses tailored to the user's needs.

#### Efficient Memory Management:
Dividing the text into chunks avoids running into token limits of LLMs, allowing the model to process large documents effectively without losing important details from the encyclopedia.

#### Improved User Experience:
With precise, well-formatted responses, GaleMed Insights enhances the readability and accessibility of complex medical information, making it easier for users to understand and act on the information provided.

---
## Why This Project?
The GaleMed Insights project is designed to address the growing need for reliable, accurate, and accessible medical information. In a world where misinformation spreads easily, especially on health-related topics, GaleMed Insights offers a trusted source of knowledge from The Gale Encyclopedia of Medicine. Key reasons for developing this project include:

#### Improving Health Literacy:
GaleMed Insights empowers individuals to make informed decisions about their health by providing clear and factual medical information.

#### Supporting Healthcare Professionals:
The chatbot serves as a quick reference tool for healthcare providers, giving them rapid access to accurate medical information to assist in patient care.

#### Easy Knowledge Base Updates:
As medical knowledge evolves, GaleMed Insights can easily be updated with new data, ensuring the chatbot remains current and provides the latest medical insights.

---
## Steps: 
### Document Reading with PyPDF:
The project utilizes the pyPDF library to read and extract data from The Gale Encyclopedia of Medicine. This library efficiently handles large documents, facilitating the structured extraction of valuable medical information.

### Data Chunking:
After extracting the data, it is crucial to divide the information into manageable chunks. Chunking helps to:

- Improve retrieval efficiency by enabling quick access to relevant sections.

- Enhance context provided in responses, allowing the model to generate more meaningful answers.

- Mitigate issues related to the maximum token limits of language models by ensuring that only concise and relevant information is processed at any given time.

### Creating Semantic Embeddings:

- To facilitate meaningful queries and responses, we employ the HuggingFaceEmbeddings model (sentence-transformers/all-MiniLM-L6-v2). This model generates semantic embeddings, which capture the context and meaning of the text beyond surface-level words. These embeddings enable GaleMed to comprehend user queries more effectively and retrieve the most relevant information.

### Building a Vector Database with Pinecone:
The next step involves creating a vector database using Pinecone. We load the chunked and embedded data into Pinecone, establishing a knowledge base that can efficiently manage user queries by identifying similar vectors and retrieving pertinent information.

### Testing Queries:
Once the knowledge base is established, we conduct test queries to evaluate the performance of GaleMed. This testing phase ensures that the chatbot retrieves accurate and relevant information effectively, confirming the effectiveness of the RAG approach.

### Utilizing LLM for Response Formatting:
To enhance the clarity and readability of the information returned to users, we utilize a large language model (LLM) based on the LLaMA architecture (CTransformers). This offline version processes the extracted data and formats it into coherent responses, making complex medical information accessible and understandable.

### Flask Application Development:
Finally, we aim to develop a Flask web application that serves as the interface for GaleMed. This application will allow users to interact with the chatbot seamlessly, posing questions and receiving comprehensive answers based on the medical encyclopedia.

---
## Real-World Use Cases of RAG Methodology in Other Fields
This RAG-based approach is not only useful in healthcare but also has real-world applications in various fields, such as:

#### Legal Industry:
Legal research chatbots can use RAG to pull up relevant laws, precedents, and case summaries from specific legal databases, providing accurate and contextual legal advice.

#### Customer Support:
Companies can use RAG to power customer service bots that retrieve relevant product information and troubleshooting steps from their internal knowledge bases, enhancing customer experience with quicker and more accurate responses.

#### Education:
RAG-based educational bots can assist students by providing targeted explanations and detailed answers based on textbooks, enhancing their learning experiences.

#### Technical Documentation:
Tech companies can use RAG to help engineers and developers quickly retrieve information from large technical manuals or knowledge bases to solve problems efficiently.

---

By using a RAG framework such as GaleMed Insights, we can enable and ensure that users get precise, relevant information from a trusted source, improving the overall accuracy and reliability of responses across various industries.

Data got from https://www.academia.edu/32752835/The_GALE_ENCYCLOPEDIA_of_MEDICINE_SECOND_EDITION

In [1]:
print("OK!")

OK!


In [1]:
import os
import time
from pinecone import Pinecone, ServerlessSpec
from langchain import PromptTemplate
from langchain.chains import RetrievalQA
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore as LangchainPinecone  # Updated import
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.prompts import PromptTemplate
from langchain.llms import CTransformers


/Users/parthawgoswami/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/parthawgoswami/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/parthawgoswami/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Initializing index name and the Pinecone

os.environ["PINECONE_API_KEY"] = "pcsk_4ArCC1_Rzav3ZitfjUHYBhYFxYqkKBfVtqDyWUvH3ozBWD46nFViEpjBdx9djfBJJh8FLc"

index_name="rag-knowledge"

# Initialize Pinecone with optional parameters
try:
    pc = Pinecone(
        api_key=os.environ.get("PINECONE_API_KEY"),
        proxy_url=None,            # Example optional parameter
        proxy_headers=None,        # Example optional parameter
        ssl_ca_certs=None,        # Example optional parameter
        ssl_verify=True,  # Example optional parameter, usually set to True
    )
    
    # Check if the index exists
    indexes = pc.list_indexes()  # List of index names
    index_names = indexes.names()  # Get only the names of the indexes

    if index_name not in index_names:
        print(f'{index_name} does not exist')
        # Change the following line to create the index of your choice
        pc.create_index(
             name=index_name,
             dimension=384,
             metric="cosine",
             spec=ServerlessSpec(
                 cloud="aws",
                 region="us-east-1"
             )
         )
    else:
        print(f'{index_name} exists.')

    # Connect to the existing index
    index = pc.Index(index_name)

except Exception as e:
    print(f"An error occurred while checking indexes: {e}")

# Embedding the text chunks and storing them in Pinecone
try:
    docsearch = LangchainPinecone.from_texts(
        texts=[t.page_content for t in text_chunks],  # Assuming `text_chunks` is a list of text splits
        embedding=embeddings,  # Embedding model instance
        index_name=index_name
    )
except Exception as e:
    print(f"An error occurred while creating embeddings: {e}")

rag-knowledge exists.
An error occurred while creating embeddings: name 'text_chunks' is not defined


In [3]:
# from pinecone import Pinecone

pc = Pinecone(api_key="pcsk_4ArCC1_Rzav3ZitfjUHYBhYFxYqkKBfVtqDyWUvH3ozBWD46nFViEpjBdx9djfBJJh8FLc")
pc.list_indexes()

[
    {
        "name": "rag-knowledge",
        "metric": "cosine",
        "host": "rag-knowledge-oaavv06.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 1024,
        "deletion_protection": "disabled",
        "tags": null,
        "embed": {
            "model": "llama-text-embed-v2",
            "field_map": {
                "text": "text"
            },
            "dimension": 1024,
            "metric": "cosine",
            "write_parameters": {
                "dimension": 1024.0,
                "input_type": "passage",
                "truncate": "END"
            },
            "read_parameters": {
                "dimension": 1024.0,
                "input_type": "query",
                "truncate": "END"
    

### Vector stores are there in the Pinecone that we created

In [2]:
#Extract data from the PDF
def load_pdf(data):
    loader = DirectoryLoader(data,
                    glob="*.pdf",
                    loader_cls=PyPDFLoader)
    
    documents = loader.load()

    return documents

In [3]:
extracted_data = load_pdf("/Users/parthawgoswami/Documents/ADV_NLP/Medical_books/")

In [6]:
#extracted_data

In [4]:
#Create text chunks
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 20)
    text_chunks = text_splitter.split_documents(extracted_data)

    return text_chunks

In [5]:
text_chunks = text_split(extracted_data)
print("length of my chunk:", len(text_chunks))

length of my chunk: 1173


In [9]:
# text_chunks

In [6]:
#download embedding model
def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [7]:
embeddings = download_hugging_face_embeddings()

/var/folders/gx/_792gs0d1p5_1_15m1lwtn9m0000gn/T/ipykernel_67564/1337643473.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [12]:
embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [13]:
query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

Length 384


In [14]:
# query_result

### Writing the Data into Pinecone

In [9]:
#Initializing index name and the Pinecone

os.environ["PINECONE_API_KEY"] = "pcsk_4ArCC1_Rzav3ZitfjUHYBhYFxYqkKBfVtqDyWUvH3ozBWD46nFViEpjBdx9djfBJJh8FLc"

index_name="rag-knowledge"

# Initialize Pinecone with optional parameters
try:
    pc = Pinecone(
        api_key=os.environ.get("PINECONE_API_KEY"),
        proxy_url=None,            # Example optional parameter
        proxy_headers=None,        # Example optional parameter
        ssl_ca_certs=None,        # Example optional parameter
        ssl_verify=True,  # Example optional parameter, usually set to True
    )
    
    time.sleep(2)  # Optional sleep to ensure initialization completes

    # Check if the index exists
    indexes = pc.list_indexes()  # List of index names
    index_names = indexes.names()  # Get only the names of the indexes

    if index_name not in index_names:
        print(f'{index_name} does not exist')
        # Optionally, create a new index
        # Uncomment the following line to create the index
        # pc.create_index(name=index_name, dimension=384, metric='cosine')
    else:
        print(f'{index_name} exists.')

    # Connect to the existing index
    index = pc.Index(index_name)

except Exception as e:
    print(f"An error occurred while checking indexes: {e}")

# Embedding the text chunks and storing them in Pinecone
#try:
#    docsearch = LangchainPinecone.from_texts(
#        texts=[t.page_content for t in text_chunks],  # Assuming `text_chunks` is a list of text splits
#        embedding=embeddings,  # Embedding model instance
#        index_name=index_name
#    )
#except Exception as e:
#    print(f"An error occurred while creating embeddings: {e}")

rag-knowledge exists.


### Testing it out

In [10]:
index_name="rag-knowledge"
#If we already have an index we can load it like this
docsearch=LangchainPinecone.from_existing_index(index_name, embeddings)

query = "What are Allergies"

docs=docsearch.similarity_search(query, k=3)

print("Result", docs)

Result []


In [18]:
query = "Cure for acne?"

docs=docsearch.similarity_search(query, k=3)

print("Result", docs)

Result [Document(metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 39.0, 'page_label': '40', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': '/Users/parthawgoswami/Documents/ECHO_Cases/RAG_based_schema_matching/MatchMaker/Medical_book.pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'), Document(metadata={}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'), Document(metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 39.0, 'page_label': '40', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': '/Users/parthawgoswami/Documents/ECHO_Cases/RAG_based_schema_matching/MatchMaker/Medical_book.pdf', 'total_pages': 637.0}, page_content='milk thistle (Silybum marianum), and with nutrients such\nas essential fatty a

In [19]:
query = "I have pain in my head"

docs=docsearch.similarity_search(query, k=3)

print("Result", docs)

Result [Document(metadata={}, page_content='of the brain. They can rupture, causing subarachnoid hemorrhage.CLINICAL APPROACHHe a da c he  i s  one  of the  mos t c ommon c ompl a i nts  of pa ti e nts  i n me di c a l  pra c ti c e .  It periodically afflicts 90% of adults, and almost 25% have recurrent severe head-aches. As with many common symptoms, a broad range of conditions, from trivial to life-threatening, might be responsible. The majority of patients presenting with headache have tension-type, migraine, or cluster; however,'), Document(metadata={}, page_content='A 5 9 -ye a r-o ld  w o m a n  co m e s t o  yo u r clin ic b e ca u se  sh e  is co n ce rn e d  t h a t  sh e  might have a brain tumor. She has had a fairly severe headache for the last  3 weeks (she rates it as an 8 on a scale of 1-10). She describes the pain as constant, occasionally throbbing but mostly a dull ache, and localized to the right side of her head. She thinks the pain is worse at night, especially wh

In [20]:
docs

[Document(metadata={}, page_content='of the brain. They can rupture, causing subarachnoid hemorrhage.CLINICAL APPROACHHe a da c he  i s  one  of the  mos t c ommon c ompl a i nts  of pa ti e nts  i n me di c a l  pra c ti c e .  It periodically afflicts 90% of adults, and almost 25% have recurrent severe head-aches. As with many common symptoms, a broad range of conditions, from trivial to life-threatening, might be responsible. The majority of patients presenting with headache have tension-type, migraine, or cluster; however,'),
 Document(metadata={}, page_content='A 5 9 -ye a r-o ld  w o m a n  co m e s t o  yo u r clin ic b e ca u se  sh e  is co n ce rn e d  t h a t  sh e  might have a brain tumor. She has had a fairly severe headache for the last  3 weeks (she rates it as an 8 on a scale of 1-10). She describes the pain as constant, occasionally throbbing but mostly a dull ache, and localized to the right side of her head. She thinks the pain is worse at night, especially when she

In [21]:
query = "I am sad all the time"

docs=docsearch.similarity_search(query, k=3)

print("Result", docs)

Result [Document(metadata={}, page_content='has been tearful on occasions. He has lost interest in his hobbies and has not attempted to \nhave sexual intercourse with his wife. He feels very tired and has problems concentrating \nwhen watching the television or reading the newspaper. He has lost about half a stone since \nthe operation despite the lack of physical exercise. He admits to suicidal thoughts, but denies \nany concrete plans. Asked to compare his emotional state with a previous episode of depres -'), Document(metadata={}, page_content='sons or objects outside the self that persists despite\nthe facts.\nDepression—A state of being depressed marked\nespecially by sadness, inactivity, difficulty with\nthinking and concentration, a significant increase or\ndecrease in appetite and time spent sleeping, feel-\nings of dejection and hopelessness, and sometimes\nsuicidal thoughts or an attempt to commit suicide.\nGlucocorticoid—Any of a group of corticosteroids\n(as hydrocortisone 

In [22]:
query = "What to do if you are sad?"

docs=docsearch.similarity_search(query, k=3)

print("Result", docs)

Result [Document(metadata={}, page_content='diagnosed mild depression and offered therapy, but the girl failed to attend further sessions \nand was subsequently discharged back to the GP.\nExamination\nOn examination, the girl is well-groomed, quiet with her gaze lowered. Her answers to ques -\ntions on her well-being are monosyllabic. She denies suicidal intent or plans to harm herself. \nEnquiries on her daily activities are unfruitful as she does not engage in any conversation.'), Document(metadata={}, page_content='knees and wrists are painful at all times and he has increasing difficulty in doing his work. \nHe is feeling very exhausted and tired, his mood is low and he becomes irritable very eas-\nily. He has been crying unprovoked. He admits to problems falling asleep at night and early \nmorning waking. He has daily thoughts of life not being worth living, but denies any suicidal \nthoughts or intent. He has suffered two bereavements recently: a good friend dying after a'), Doc

In [23]:
query = "Who is superman?"

docs=docsearch.similarity_search(query, k=3)

print("Result", 
      docs)

Result [Document(metadata={}, page_content='yourself.\nKey Points'), Document(metadata={}, page_content='subsequent career.\nP John Rees\nJames Pattison\nGwyn Williams'), Document(metadata={}, page_content='xv')]
